In [ ]:
import os
import sys
import numpy as np
import pandas as pd

sys.path.append(os.path.abspath("../../")) ; from EPF import variables
sys.path.insert(0, "../helpers/"); from run_parrellel import run_parallel

Display accuracy metrics

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


def print_metrics(eval_output):
    m, n = eval_output["model"], eval_output["naive"]
    _UNITLESS = {"r2"}
    _PCT = {"mape", "wmape", "spike_pct", "dip_pct"}
    for metric, val in m.items():
        if metric in _UNITLESS:
            fmt = f"{val:8.4f}"; naive_fmt = f"{n.get(metric, 0.0):8.4f}"
        elif metric in _PCT:
            fmt = f"{val:8.2f}%"; naive_fmt = f"{n.get(metric, 0.0):8.2f}%" if metric in n else "     n/a"
        elif metric in ("spike_mae", "nonspike_mae"):
            fmt = f"{val:8.2f} $/MWh"; naive_fmt = "     n/a"
        else:
            fmt = f"{val:8.2f} $/MWh"; naive_fmt = f"{n.get(metric, 0.0):8.2f} $/MWh"
        print(f"  {metric.upper():14s}  LGBM: {fmt}   Naive: {naive_fmt}", flush=True)
    if n.get("mae", 0) > 0:
        print(f"  {'SKILL':14s}  MAE skill vs naive: {(1 - m['mae'] / n['mae']) * 100:+.2f}%", flush=True)
    steps_df = eval_output.get("steps_df")
    if steps_df is not None:
        print(f"\n  {'step':>4}  {'lead':>6}  {'MAE':>8}  {'RMSE':>8}  {'R²':>7}  {'MBE':>8}", flush=True)
        for _, row in steps_df.iterrows():
            print(f"  step {int(row['step']):>3}  lead={row['lead_h']:.1f}h  MAE={row['mae']:>7.2f}  RMSE={row['rmse']:>7.2f}  R²={row['r2']:>6.4f}  MBE={row['mbe']:>+7.2f}", flush=True)
    fi_df = eval_output.get("feature_importance")
    if fi_df is not None:
        print(f"\n  Top 20 features by mean gain", flush=True)
        for rank, row in fi_df.head(20).iterrows():
            print(f"    {rank + 1:>3}.  {row['feature']:<45}  {row['importance']:>12.0f}")